In [1]:
# %load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
# %%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
#from tpch import load_customer, load_orders, q22
DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15" #"/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)


In [ ]:

### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
# %%time
# ### cell 2 ###

# def_codes = ["13", "31", "23", "29", "30", "18", "17"]

# # 1. extract country code and initial filters
# customer_filtered = (
#     customer[["C_CUSTKEY", "C_ACCTBAL", "C_PHONE"]]
#     .assign(CNTRYCODE=customer.C_PHONE.str.slice(0, 2))
# )
# customer_filtered = customer_filtered[
#     (customer_filtered.C_ACCTBAL > 0.0)
#     & (customer_filtered.CNTRYCODE.isin(def_codes))
# ]

# # 2. filter above-average balances
# avg_balance = customer_filtered.C_ACCTBAL.mean()
# customer_filtered = customer_filtered[customer_filtered.C_ACCTBAL > avg_balance]

# # 3. anti-join: drop customers with any orders
# customer_filtered = customer_filtered[~customer_filtered.C_CUSTKEY.isin(orders.O_CUSTKEY)]

# # 4. single groupby for both aggregations
# total = (
#     customer_filtered
#     .groupby("CNTRYCODE")
#     .agg({"C_CUSTKEY": "count", "C_ACCTBAL": "sum"})
#     .reset_index()
#     .rename(columns={"C_CUSTKEY": "NUMCUST", "C_ACCTBAL": "TOTACCTBAL"})
#     .sort_values("CNTRYCODE")
# )


In [ ]:
# total,1.76

In [ ]:
%%time
#original
customer_filtered = customer.loc[:, ["C_ACCTBAL", "C_CUSTKEY"]]
customer_filtered["CNTRYCODE"] = customer["C_PHONE"].str.slice(0, 2)
customer_filtered = customer_filtered[
    (customer["C_ACCTBAL"] > 0.00)
    & customer_filtered["CNTRYCODE"].isin(
        ["13", "31", "23", "29", "30", "18", "17"]
    )
]
avg_value = customer_filtered["C_ACCTBAL"].mean()
customer_filtered = customer_filtered[customer_filtered["C_ACCTBAL"] > avg_value]
# Select only the keys that don't match by performing a left join and only selecting columns with an na value
orders_filtered = orders.loc[:, ["O_CUSTKEY"]].drop_duplicates()
customer_keys = customer_filtered.loc[:, ["C_CUSTKEY"]].drop_duplicates()
customer_selected = customer_keys.merge(
    orders_filtered, left_on="C_CUSTKEY", right_on="O_CUSTKEY", how="left"
)
customer_selected = customer_selected[customer_selected["O_CUSTKEY"].isna()]
customer_selected = customer_selected.loc[:, ["C_CUSTKEY"]]
customer_selected = customer_selected.merge(
    customer_filtered, on="C_CUSTKEY", how="inner"
)
customer_selected = customer_selected.loc[:, ["CNTRYCODE", "C_ACCTBAL"]]
agg1 = customer_selected.groupby(["CNTRYCODE"], as_index=False, sort=False).size()
agg1.columns = ["CNTRYCODE", "NUMCUST"]
agg2 = customer_selected.groupby(["CNTRYCODE"], as_index=False, sort=False).agg(
    TOTACCTBAL=pd.NamedAgg(column="C_ACCTBAL", aggfunc="sum")
)
total = agg1.merge(agg2, on="CNTRYCODE", how="inner")
total = total.sort_values(by=["CNTRYCODE"], ascending=[True])


In [8]:
total

,CNTRYCODE,NUMCUST,TOTACCTBAL
3,13,9025,67592468.28
6,17,9067,68084663.34
1,18,9210,69312783.61
2,23,8984,67607771.32
5,29,9199,69015438.26
0,30,9343,70118838.04
4,31,9086,68144525.38


In [9]:
### cell 3 ###

total.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 7 entries, 3 to 4
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   CNTRYCODE   7 non-null      object
 1   NUMCUST     7 non-null      int64
 2   TOTACCTBAL  7 non-null      float64
dtypes: float64(1), int64(1), object(1)
memory usage: 214.0+ bytes
